In [17]:
import sys
from pathlib import Path

import pandas as pd

# Resolve project root whether the notebook is run from notebooks/ or repo root
_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if not (_cwd / "data").exists() and (_cwd.parent / "data").exists() else _cwd

sys.path.insert(0, str(PROJECT_ROOT))

MASTER_PATH = PROJECT_ROOT / "data" / "processed" / "mibi_cellData_with_patient_class_and_centroids.csv"
SPLIT_OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "mibi_cells_with_splits.csv"
SPLIT_SUMMARY_PATH = PROJECT_ROOT / "reports" / "tables" / "split_summary.csv"

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "reports" / "tables"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Master table:", MASTER_PATH)
print("Split output:", SPLIT_OUTPUT_PATH)
print("Summary output:", SPLIT_SUMMARY_PATH)

Project root: c:\Users\Kelly\Documents\spatial-ihc-feature-lab
Master table: c:\Users\Kelly\Documents\spatial-ihc-feature-lab\data\processed\mibi_cellData_with_patient_class_and_centroids.csv
Split output: c:\Users\Kelly\Documents\spatial-ihc-feature-lab\data\processed\mibi_cells_with_splits.csv
Summary output: c:\Users\Kelly\Documents\spatial-ihc-feature-lab\reports\tables\split_summary.csv


In [18]:
from src.utils.splits import (
    make_stratified_group_split_for_group_target,
    split_summary,
)

In [19]:
# 1. Load your merged MIBI table
if not MASTER_PATH.exists():
    raise FileNotFoundError(f"Missing file: {MASTER_PATH}")

df = pd.read_csv(MASTER_PATH)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Loaded 197,678 rows x 61 columns


,SampleID,cellLabelInImage,cellSize,C,Na,Si,P,Ca,Fe,dsDNA,...,Au,tumorYN,tumorCluster,Group,immuneCluster,immuneGroup,patient_class,centroid_x,centroid_y,mask_area_pixels
0,1,2,146,0,-0.59061,0.875220,-2.57650,-0.806060,-0.2344,-1.308000,...,2.32660,1,0,6,0,0,0,32.856164,158.335616,146.0
1,1,3,102,0,-0.49870,0.017464,-1.22490,-0.501520,-1.3412,0.522570,...,2.18710,0,0,2,46,4,0,30.588235,191.294118,102.0
2,1,4,43,0,-1.48730,-0.630440,-1.91070,-1.228000,-1.3937,-1.782200,...,1.47820,1,0,6,0,0,0,30.232558,212.325581,43.0
3,1,5,211,0,-1.00530,-0.532270,-1.74300,-0.944850,-1.0996,-0.057906,...,1.58620,1,0,6,0,0,0,34.772512,269.985782,211.0
4,1,6,177,0,0.15803,-0.710290,0.51737,-0.096251,-1.0526,0.355020,...,-0.49627,0,0,2,75,6,0,35.344633,381.175141,177.0


In [20]:
# 2. Create train/val/test split by patient
df_split = make_stratified_group_split_for_group_target(
    df=df,
    group_col="SampleID",          # patient/sample column
    target_col="patient_class",    # 0/1/2 mixed/compartmentalized/cold
    test_size=0.20,
    val_size=0.20,
    random_state=42,
)

In [21]:
# 3. Create split summary table
summary = split_summary(
    df_split,
    split_col="split",
    group_col="SampleID",
    target_col="patient_class",
)

In [22]:
# 4. Save outputs
df_split.to_csv(SPLIT_OUTPUT_PATH, index=False)
summary.to_csv(SPLIT_SUMMARY_PATH, index=False)

print("Saved:")
print(SPLIT_OUTPUT_PATH)
print(SPLIT_SUMMARY_PATH)
print()
print(summary)

Saved:
c:\Users\Kelly\Documents\spatial-ihc-feature-lab\data\processed\mibi_cells_with_splits.csv
c:\Users\Kelly\Documents\spatial-ihc-feature-lab\reports\tables\split_summary.csv

   split  n_rows  n_groups  class_0_groups  class_1_groups  class_2_groups
0   test   44697         8               4               3               1
1  train  114670        24              11               9               4
2    val   38311         8               4               3               1
